# Rotating-blade Farassat 1A showcase

This notebook reads the pressure stored on the solid blade surface in `p_surface_10x7E_7000RPM_AMI.npz` and evaluates the moving, impermeable Farassat 1A formulation. It previews the blade data, predicts observer pressure, checks the fully arrived record, and plots spectra and directivity.

## Scope and formulation

The archive is the legacy `data`/`columns` schema. Its pressure is OpenFOAM kinematic pressure, so the reader multiplies it by $\rho=1.225\,\mathrm{kg/m^3}$ and removes the configured reference pressure. The archive normals are reversed for the acoustic convention.

`permeable=False` selects the solid, impermeable formulation: no fluid-velocity `U` field is needed. `moving_surface=True` enables time-dependent geometry, source Mach/Doppler terms, and moving retarded times. The reference face centers and normals are rigidly rotated about the global $y$ axis, while surface velocity and acceleration are reconstructed from $\boldsymbol{\omega}\times\mathbf r$. The solver retains both loading and thickness contributions.

Acoustic propagation assumes constant density and sound speed, zero mean-flow convection, and neglects quadrupole volume sources. Every moving face must remain subsonic.

In [ ]:
import importlib
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from foamacoustics import F1ASolver, NPZSurfaceSeries
from foamacoustics.plotting import (
    plot_oaspl_directivity,
    plot_pressure_history,
    plot_spl_spectrum,
    plot_surface_pressure_preview,
)
from foamacoustics.utils import (
    spl_spectrum_to_a_weighted_oaspl,
    spl_spectrum_to_oaspl,
)

importlib.import_module("scienceplots")
plt.style.use(["science", "no-latex"])

## 1. Configure the blade case and observers

The explicit three-component angular velocity is important: this legacy archive stores only the scalar magnitude `omega`, which does not identify the rotation axis. The seven default observers keep the showcase manageable; extend `observer_angles_deg` for a finer directivity sweep.

In [ ]:
blade_npz = Path("/home/dpusino/p_surface_10x7E_7000RPM_AMI.npz")
if not blade_npz.is_file():
    raise FileNotFoundError(blade_npz)

rpm = 7000.0
blade_count = 2
rotation_frequency_hz = rpm / 60.0
omega_rad_s = torch.tensor(
    [0.0, 2.0 * torch.pi * rotation_frequency_hz, 0.0],
    dtype=torch.float64,
)
rotation_center_m = [0.0, 0.0, 0.0]

density_kg_m3 = 1.225
sound_speed_m_s = 343.0
pressure_reference_pa = 101325.0 * density_kg_m3
response_delay_s = 1.0e-5
numerical_dtype = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
chunk_size = 64

observer_radius_m = 1.88
observer_angles_deg = torch.arange(
    90.0, -91.0, -10.0, device=device, dtype=numerical_dtype
)
observer_angles_rad = torch.deg2rad(observer_angles_deg)
observers_m = observer_radius_m * torch.stack(
    (
        torch.cos(observer_angles_rad),
        torch.sin(observer_angles_rad),
        torch.zeros_like(observer_angles_rad),
    ),
    dim=1,
)

print(f"Archive: {blade_npz}")
print(f"Device: {device}; dtype: {numerical_dtype}; chunk size: {chunk_size}")
print(f"Angular velocity: {omega_rad_s.tolist()} rad/s")
print(f"Observers: {len(observers_m)} at radius {observer_radius_m:g} m")

## 2. Load and validate the legacy blade archive

Loading streams the large compressed `data.npy` member once, retaining the first reference geometry and the selected `p_surface_true` history. For this rigid rotor, later face geometry is reconstructed analytically instead of retaining a full geometry copy at every time.

In [ ]:
load_start = perf_counter()
series = NPZSurfaceSeries(
    blade_npz,
    pressure_field="p_surface_true",
    pressure_units="kinematic",
    density_kg_m3=density_kg_m3,
    pressure_reference_pa=pressure_reference_pa,
    rotation_center_m=rotation_center_m,
    omega_rad_s=omega_rad_s,
    flip_normals=True,
    dtype=numerical_dtype,
)
first_frame = next(series.iter_frames())
time_steps_s = torch.diff(series.absolute_times_s)
surface_speed_m_s = torch.linalg.vector_norm(first_frame.surface_velocity_m_s, dim=1)

assert first_frame.fluid_velocity_m_s is None
assert first_frame.surface_velocity_m_s is not None
assert first_frame.surface_acceleration_m_s2 is not None
assert torch.isfinite(first_frame.pressure_pa).all()
assert float(surface_speed_m_s.max()) < sound_speed_m_s

print(f"Schema: {series.schema}; field: {series.pressure_field}")
print(f"Loaded in {perf_counter() - load_start:.2f} s")
print(f"Frames: {len(series):,}; faces: {series.panel_count:,}")
print(f"Source times: {series.absolute_times_s[0]:.7f} to {series.absolute_times_s[-1]:.7f} s")
print(f"Duration: {float(series.source_times_s[-1]):.9f} s")
print(f"Time-step range: {float(time_steps_s.min()):.9g} to {float(time_steps_s.max()):.9g} s")
print(f"Total blade area: {float(series.face_areas_m2.sum()):.9g} m^2")
print(f"First-frame gauge-pressure range: {float(first_frame.pressure_pa.min()):.6g} to {float(first_frame.pressure_pa.max()):.6g} Pa")
print(f"Maximum face speed: {float(surface_speed_m_s.max()):.6g} m/s (Mach {float(surface_speed_m_s.max()) / sound_speed_m_s:.4f})")

In [ ]:
plot_surface_pressure_preview(
    first_frame.face_centers_m,
    first_frame.pressure_pa,
    title="10x7E blade surface — first-frame gauge pressure",
)
plt.show()

## 3. Configure the moving impermeable F1A solver

The formulation choice is explicit and independent of the file format. The source pressure supplies $\mathbf L=p'\mathbf n$; rigid-body motion supplies the normal surface velocity used by the thickness term and all moving-source kinematics. Moving arrival times always use the nonuniform propagation path.

In [ ]:
solver = F1ASolver(
    series,
    rpm=rpm,
    permeable=False,
    moving_surface=True,
    rotation_center_m=rotation_center_m,
    omega_rad_s=omega_rad_s,
    density_kg_m3=density_kg_m3,
    sound_speed_m_s=sound_speed_m_s,
    response_delay_s=response_delay_s,
    adjust_time_step=True,
    device=device,
    chunk_size=chunk_size,
)

print(f"Formulation: moving={solver.moving_surface}, permeable={solver.permeable}")
print(f"Panels: {solver.panel_count:,}; observer angles: {observer_angles_deg.detach().cpu().tolist()} deg")

## 4. Predict observer pressure

This is the expensive cell. It streams pressure in chunks, reconstructs the blade state at every source time, evaluates the moving-surface thickness and loading terms, and deposits them at their face-dependent observer arrival times.

In [ ]:
progress = tqdm(total=len(series), desc="Moving blade F1A", unit="frame")

def report_progress(done, total, message):
    del total, message
    progress.update(done - progress.n)
    if done == len(series):
        progress.close()

solve_start = perf_counter()
result = solver.predict(
    observers_m,
    progress_callback=report_progress,
    progress_every=25,
    remove_mean=False,
    algorithm="auto",
)
elapsed_s = perf_counter() - solve_start

assert result.total_pressure_pa.shape == (len(observers_m), len(series))
assert torch.isfinite(result.total_pressure_pa).all()
torch.testing.assert_close(
    result.total_pressure_pa,
    result.thickness_pressure_pa + result.loading_pressure_pa,
)

print(f"Propagation algorithm: {result.propagation_algorithm}")
print(f"Output shape: {tuple(result.total_pressure_pa.shape)}")
print(f"Compute device: {result.device}; wall time: {elapsed_s:.2f} s")

## 5. Inspect arrivals and pressure components

Only samples after an observer's maximum face delay contain contributions from the entire moving blade. The shaded part of the trace is incomplete and must not be included in a quantitative spectrum.

In [ ]:
maximum_delays_s = torch.as_tensor(
    result.maximum_delay_s,
    device=result.observer_times_s.device,
).reshape(-1)
fully_arrived = result.observer_times_s[None, :] >= maximum_delays_s[:, None]
if not torch.all(torch.any(fully_arrived, dim=1)):
    raise RuntimeError("The source record ends before every observer has a fully arrived sample.")

for index, angle_deg in enumerate(observer_angles_deg.detach().cpu()):
    tail = result.total_pressure_pa[index, fully_arrived[index]]
    rms_pa = torch.sqrt(torch.mean((tail - tail.mean()) ** 2))
    print(
        f"{float(angle_deg):+5.0f} deg: delay={float(maximum_delays_s[index]) * 1e3:.4f} ms, "
        f"fully arrived samples={tail.numel():,}, RMS={float(rms_pa):.6g} Pa"
    )

history_observer_index = int(torch.argmin(torch.abs(observer_angles_deg)))
plot_pressure_history(
    result.for_observer(history_observer_index),
    annotate_arrivals=True,
    title=f"Rotating solid-blade F1A — {float(observer_angles_deg[history_observer_index]):+g}° observer",
)
plt.show()

## 6. Compute fully arrived spectra

The final nine rotor rotations are used, matching the existing propeller comparison scripts. The helper resamples the slightly nonuniform source-time record before applying the FFT and marks blade-passing-frequency harmonics.

In [ ]:
requested_rotations = 9
spectrum_frequency_ceiling_hz = 5000.0
source_duration_s = float(series.source_times_s[-1])
required_duration_s = requested_rotations / rotation_frequency_hz + float(maximum_delays_s.max())

print(f"Available duration: {source_duration_s:.6f} s ({source_duration_s * rotation_frequency_hz:.3f} rotations)")
print(f"Required duration:  {required_duration_s:.6f} s")
if source_duration_s < required_duration_s:
    raise RuntimeError(
        f"The record is too short for {requested_rotations} fully arrived rotations."
    )

spectrum_plot = plot_spl_spectrum(
    result,
    rotations=requested_rotations,
    blade_count=blade_count,
    xlim_hz=(None, spectrum_frequency_ceiling_hz),
)
frequency_hz = spectrum_plot.frequencies_hz
spl_db = spectrum_plot.spl_db
print(f"Frequency resolution: {spectrum_plot.frequency_resolution_hz:.6g} Hz")
plt.show()

## 7. Plot OASPL directivity

DC is excluded and the spectrum is integrated through 5 kHz for both unweighted and A-weighted overall levels. Add more observer angles in the configuration cell when a smoother directivity curve is needed.

In [ ]:
oaspl_db = spl_spectrum_to_oaspl(
    spl_db[:, 1:],
    frequency_hz[1:],
    frequency_ceiling_hz=spectrum_frequency_ceiling_hz,
    dim=1,
)
a_weighted_oaspl_db = spl_spectrum_to_a_weighted_oaspl(
    spl_db[:, 1:],
    frequency_hz[1:],
    frequency_ceiling_hz=spectrum_frequency_ceiling_hz,
    dim=1,
)
plot_oaspl_directivity(
    observer_angles_deg.detach().cpu(),
    oaspl_db,
    a_weighted_oaspl_db=a_weighted_oaspl_db,
)
plt.show()

for angle_deg, level_db, level_a_db in zip(
    observer_angles_deg.detach().cpu(),
    oaspl_db.detach().cpu(),
    a_weighted_oaspl_db.detach().cpu(),
):
    print(f"{float(angle_deg):+5.0f} deg: {float(level_db):7.3f} dB, {float(level_a_db):7.3f} dBA")